# AutoCharter — Inferencia en canción nueva

Pipeline completo desde un archivo de audio hasta un `.zip` listo para Clone Hero.

**Pasos:**
1. Configuración y carga del modelo
2. Separación de stems (Demucs) por instrumento
3. Estimación de beats y extracción de MERT + Log-Mel por beat
4. Inferencia en batch (multi-instrumento × multi-dificultad)
5. Decodificación de tokens → `ChartData` por track
6. Ensamblado del `notes.chart` multi-track
7. Conversión de audio a OGG y generación de `song.ini`
8. Empaquetado ZIP y descarga

## 1 · Configuración

In [ ]:
import sys, json, zipfile, shutil, subprocess
from pathlib import Path

REPO_ROOT      = Path("/s/mlopezs8/CloneCharter")
CHECKPOINT_DIR = REPO_ROOT / "checkpoints/run1/best"
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Canción de entrada ────────────────────────────────────────────────────────
AUDIO_PATH = Path("/path/to/song.ogg")   # ← cambiar a la ruta del archivo de audio

# ── Metadatos ─────────────────────────────────────────────────────────────────
SONG_NAME  = "Mi Canción"
ARTIST     = "Artista"
ALBUM      = ""
GENRE      = "Rock"
YEAR       = 2024
OUTPUT_DIR = REPO_ROOT / "output_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Instrumentos y dificultades a generar ─────────────────────────────────────
# Lista de (instrumento, dificultad_id) a generar en batch.
# difficulty_id: 0=Easy, 1=Medium, 2=Hard, 3=Expert
TARGETS = [
    ("guitar", 3),   # Guitar Expert
    ("guitar", 2),   # Guitar Hard
    ("bass",   3),   # Bass Expert
    ("drums",  3),   # Drums Expert
]

# ── Parámetros de decodificación ──────────────────────────────────────────────
# Nucleus sampling: buena calidad sin ser determinista
TEMPERATURE    = 0.95
TOP_P          = 0.92
TOP_K          = 0        # 0 = desactivado (usar nucleus)
MAX_NEW_TOKENS = 8192     # máximo tokens por track

_INSTR_TO_ID = {"guitar": 0, "bass": 1, "drums": 2}
_DIFF_NAMES  = {0: "Easy", 1: "Medium", 2: "Hard", 3: "Expert",
                4: "Expert+", 5: "Expert++", 6: "Expert+++"}
print("Configuración lista.")

## 2 · Cargar el modelo

In [ ]:
from auto_charter.model.charter_model import AutoCharterModel

model = AutoCharterModel.from_pretrained(CHECKPOINT_DIR)
model = model.to(DEVICE)
model.eval()
print(f"Modelo: {model.num_parameters():,} parámetros | best_val_loss={json.loads((CHECKPOINT_DIR/'trainer_state.json').read_text())['best_val_loss']:.4f}")

## 3 · Separación de stems con Demucs

Se separa el audio en stems (guitar/other, bass, drums) usando Demucs `htdemucs`. Solo se procesan los instrumentos presentes en `TARGETS`.

In [ ]:
from auto_charter.audio.separator import StemSeparator

# Instrumentos únicos necesarios para los TARGETS
needed_instruments = list({instr for instr, _ in TARGETS})

stems_dir = OUTPUT_DIR / "stems"
stems_dir.mkdir(parents=True, exist_ok=True)

print(f"Separando stems: {needed_instruments} ...")
try:
    stem_paths = StemSeparator(device=DEVICE).separate(
        AUDIO_PATH, stems_dir, instruments=needed_instruments
    )
    print("Stems separados:")
    for instr, path in stem_paths.items():
        print(f"  {instr}: {path}")
except Exception as e:
    print(f"⚠ Demucs falló ({e}). Usando audio original para todos los instrumentos.")
    stem_paths = {instr: AUDIO_PATH for instr in needed_instruments}

## 4 · Beat estimation + MERT + Log-Mel por instrumento

Para cada stem se extrae la grilla de beats y los embeddings. El resultado se cachea en memoria para no re-extraer cuando se generan múltiples dificultades del mismo instrumento.

In [ ]:
import numpy as np
from auto_charter.audio.beat_estimator import BeatEstimator
from auto_charter.audio.mert_extractor import MERTExtractor
from auto_charter.audio.logmel import LogMelExtractor

# Instanciar extractores una sola vez (cargan modelos pesados)
mert_extractor  = MERTExtractor(device=DEVICE)
logmel_extractor = LogMelExtractor()

# Cache: instrumento → dict con tensores listos para el modelo
audio_features: dict[str, dict] = {}

for instr in needed_instruments:
    stem_path = stem_paths.get(instr, AUDIO_PATH)
    print(f"\n── {instr.upper()} ({stem_path.name}) ──────────────────────────")

    # Beat grid
    print("  Estimando beats...")
    beat_info = BeatEstimator.estimate(stem_path)
    N = beat_info["num_beats"]
    bpm = beat_info["bpm_mean"]
    print(f"  {N} beats | BPM≈{bpm:.1f}")

    # MERT
    print(f"  MERT ({N} beats)...")
    mert_arr = mert_extractor.extract_per_beat(
        stem_path,
        beat_info["beat_times_s"],
        beat_info["beat_durations_s"],
    )   # [N, 1024]

    # Log-Mel
    print("  Log-Mel...")
    logmel_arr = logmel_extractor.extract_per_beat(
        stem_path,
        beat_info["beat_times_s"],
        beat_info["beat_durations_s"],
    )   # [N, 32, 128]

    # Alinear longitudes (pueden diferir ±1 beat)
    N_actual = min(mert_arr.shape[0], logmel_arr.shape[0], len(beat_info["beat_times_s"]))
    mert_arr  = mert_arr[:N_actual]
    logmel_arr = logmel_arr[:N_actual]

    beat_times      = beat_info["beat_times_s"][:N_actual]
    beat_durations  = beat_info["beat_durations_s"][:N_actual]
    bpm_at_beat     = beat_info["bpm_at_beat"][:N_actual]
    ts_num          = beat_info["time_sig_num_at_beat"][:N_actual]
    ts_den          = beat_info["time_sig_den_at_beat"][:N_actual]

    # Convertir a tensores PyTorch [1, N, ...]
    audio_features[instr] = {
        "mert":     torch.from_numpy(mert_arr).unsqueeze(0).float().to(DEVICE),
        "logmel":   torch.from_numpy(logmel_arr).unsqueeze(0).float().to(DEVICE),
        "bpm":      torch.tensor([bpm_at_beat], dtype=torch.float32, device=DEVICE),
        "ts_num":   torch.tensor([ts_num], dtype=torch.long, device=DEVICE),
        "ts_den":   torch.tensor([ts_den], dtype=torch.long, device=DEVICE),
        "dur":      torch.tensor([beat_durations], dtype=torch.float32, device=DEVICE),
        "mask":     torch.ones(1, N_actual, dtype=torch.bool, device=DEVICE),
        "bpm_mean": bpm,
        "N":        N_actual,
        "beat_times": beat_times,
    }
    print(f"  ✓ features listas ({N_actual} beats)")

print("\nExtracción completada.")

## 5 · Inferencia en batch — todos los targets

**Estrategia de decodificación:** nucleus sampling (top-p=0.92, T=0.95).

- Mejor que greedy: evita secuencias repetitivas y genera variedad rítmica más natural.
- Mejor que beam search para secuencias largas de >4000 tokens: O(k·T) vs O(k·T²).
- El encoder se evalúa **una sola vez por instrumento** y se reutiliza para todas las dificultades (la condición de instrumento y dificultad entra por los embeddings del decoder).

In [ ]:
from tqdm.auto import tqdm

# generated_tokens[(instr, diff_id)] = list[int]
generated_tokens: dict[tuple[str, int], list[int]] = {}

model.eval()
with torch.no_grad():
    for instr, diff_id in tqdm(TARGETS, desc="Generando tracks"):
        feat = audio_features[instr]
        print(f"\n  {instr} / {_DIFF_NAMES[diff_id]} — {feat['N']} beats ...")

        tokens = model.generate(
            mert_embeddings   = feat["mert"],
            logmel_frames     = feat["logmel"],
            bpm_at_beat       = feat["bpm"],
            time_sig_num      = feat["ts_num"],
            time_sig_den      = feat["ts_den"],
            beat_duration_s   = feat["dur"],
            beat_padding_mask = feat["mask"],
            instrument_id     = _INSTR_TO_ID[instr],
            difficulty_id     = diff_id,
            max_new_tokens    = MAX_NEW_TOKENS,
            temperature       = TEMPERATURE,
            top_k             = TOP_K,
            top_p             = TOP_P,
        )

        generated_tokens[(instr, diff_id)] = tokens
        print(f"  → {len(tokens)} tokens generados")

print("\n✓ Inferencia completada para todos los targets.")

## 6 · Decodificación de tokens → ChartData por track

In [ ]:
from auto_charter.tokenizer.decoder import decode_tokens
from auto_charter.parsers.chart_parser import ChartData
from auto_charter.parsers.sync_track import BPMMap, BPMEvent

# chart_data_by_instr[instr] = ChartData acumulado con todos los tracks de ese instrumento
# Usamos un ChartData por instrumento para poder hacer multi-difficulty en el mismo .chart
# (Clone Hero ignora dificultades adicionales en el mismo track; guardamos solo Expert por instr)

# Para un .chart multi-track se necesita un único ChartData que combine todos los instrumentos.
# Estrategia: usar la dificultad más alta disponible por instrumento.
_DIFF_PRIORITY = {3: 0, 2: 1, 1: 2, 0: 3}  # menor número = mayor prioridad

# Ordenar TARGETS por dificultad descendente → primer ChartData por instrumento gana
sorted_targets = sorted(TARGETS, key=lambda t: _DIFF_PRIORITY.get(t[1], 99))

combined_chart = ChartData(resolution=192)

# Construir un BPMMap común (tomamos del primer instrumento disponible)
first_instr = sorted_targets[0][0]
bpm_map = BPMMap(resolution=192)
bpm_map.bpm_events = [BPMEvent(tick=0, bpm=audio_features[first_instr]["bpm_mean"])]
combined_chart.bpm_map = bpm_map

seen_instrs: set[str] = set()   # solo el track de mayor dificultad por instrumento

for instr, diff_id in sorted_targets:
    tokens = generated_tokens[(instr, diff_id)]

    # BPMMap específico del instrumento
    instr_bpm = BPMMap(resolution=192)
    instr_bpm.bpm_events = [BPMEvent(tick=0, bpm=audio_features[instr]["bpm_mean"])]

    chart = decode_tokens(tokens, resolution=192, bpm_map=instr_bpm)

    track_notes = []
    for t_instr, notes in chart.tracks.items():
        track_notes.extend(notes)

    # Añadir al combined solo si es la dificultad más alta para este instrumento
    if instr not in seen_instrs:
        combined_chart.tracks[instr] = track_notes
        combined_chart.specials[instr] = []
        for t_instr, sp_list in chart.specials.items():
            combined_chart.specials[instr].extend(sp_list)
        seen_instrs.add(instr)
        print(f"  {instr} ({_DIFF_NAMES[diff_id]}): {len(track_notes)} notas → combinado")
    else:
        print(f"  {instr} ({_DIFF_NAMES[diff_id]}): {len(track_notes)} notas (omitido — ya hay track de mayor dificultad)")

print(f"\nTracks en combined_chart: {list(combined_chart.tracks.keys())}")

## 7 · Render notes.chart y conversión de audio a OGG

In [ ]:
import librosa
from auto_charter.parsers.chart_renderer import render_chart, render_ini

# ── notes.chart ───────────────────────────────────────────────────────────────
bpm_for_chart = audio_features[first_instr]["bpm_mean"]

chart_text = render_chart(
    combined_chart,
    bpm=bpm_for_chart,
    song_name=SONG_NAME,
    artist=ARTIST,
    album=ALBUM,
    year=YEAR,
    charter="auto-charter",
)
print(f"notes.chart: {len(chart_text)} caracteres, {len(chart_text.splitlines())} líneas")

# ── Calcular duración del audio ───────────────────────────────────────────────
y_mono, sr_mono = librosa.load(str(AUDIO_PATH), sr=22050, mono=True)
song_length_ms = int(len(y_mono) / sr_mono * 1000)
print(f"Duración del audio: {song_length_ms/1000:.1f} s")

# ── song.ini ──────────────────────────────────────────────────────────────────
# Registrar todas las dificultades más altas por instrumento
diff_by_instr = {}
for instr, diff_id in sorted_targets:
    if instr not in diff_by_instr:
        diff_by_instr[instr] = diff_id

# Construir song.ini con múltiples instrumentos
ini_lines = [
    "[Song]",
    f"name = {SONG_NAME}",
    f"artist = {ARTIST}",
]
if ALBUM:   ini_lines.append(f"album = {ALBUM}")
if GENRE:   ini_lines.append(f"genre = {GENRE}")
if YEAR:    ini_lines.append(f"year = {YEAR}")
ini_lines += [
    f"song_length = {song_length_ms}",
    "charter = auto-charter",
    f"diff_guitar = {diff_by_instr.get('guitar', -1)}",
    f"diff_bass = {diff_by_instr.get('bass', -1)}",
    f"diff_drums = {diff_by_instr.get('drums', -1)}",
    "MusicStream = song.ogg",
]
ini_text = "\n".join(ini_lines)
print(f"\nsong.ini:\n{ini_text}")

# ── Convertir audio a OGG (si aún no es .ogg) ────────────────────────────────
safe_name  = "".join(c for c in f"{ARTIST} - {SONG_NAME}" if c.isalnum() or c in " _-")[:50].strip()
build_dir  = OUTPUT_DIR / safe_name
build_dir.mkdir(parents=True, exist_ok=True)

ogg_out = build_dir / "song.ogg"
if AUDIO_PATH.suffix.lower() == ".ogg":
    shutil.copy2(AUDIO_PATH, ogg_out)
    print(f"\nAudio copiado: {ogg_out}")
else:
    print(f"\nConvirtiendo {AUDIO_PATH.suffix} → OGG con ffmpeg ...")
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(AUDIO_PATH),
         "-c:a", "libvorbis", "-q:a", "6", str(ogg_out)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"  ffmpeg stderr: {result.stderr[-500:]}")
        # Fallback: copiar el archivo original con nombre song.ogg
        shutil.copy2(AUDIO_PATH, build_dir / ("song" + AUDIO_PATH.suffix))
        ini_text = ini_text.replace("song.ogg", "song" + AUDIO_PATH.suffix)
        print("  Fallback: audio original copiado sin conversión.")
    else:
        print(f"  OGG guardado: {ogg_out}")

# Guardar chart e ini
(build_dir / "notes.chart").write_text(chart_text, encoding="utf-8")
(build_dir / "song.ini").write_text(ini_text, encoding="utf-8")
print(f"\nArchivos en: {build_dir}")

## 8 · Empaquetar ZIP y ruta de descarga

In [ ]:
zip_path = OUTPUT_DIR / f"{safe_name}.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in build_dir.iterdir():
        zf.write(f, arcname=f"{safe_name}/{f.name}")

size_mb = zip_path.stat().st_size / 1_048_576
print(f"ZIP listo: {zip_path}")
print(f"Tamaño   : {size_mb:.1f} MB")
print(f"\nContenido:")
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name}  ({info.file_size/1024:.0f} KB)")